In [2]:
!pip install -q kagglehub

import os
import glob
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy import stats
from PIL import Image
import kagglehub

# =====================================================================
# 1. DOWNLOAD DATASET & PREPARE SPLITS
# =====================================================================
print("📥 Downloading dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")

csv_search = glob.glob(f"{dataset_path}/**/*.csv", recursive=True)
df = pd.read_csv(csv_search[0])

all_image_paths = glob.glob(f"{dataset_path}/**/*.jpg", recursive=True)
path_dict = {os.path.basename(p): p for p in all_image_paths}
df['path'] = df['path'].apply(lambda x: path_dict.get(os.path.basename(str(x))))
df = df.dropna(subset=['path'])

# Keep 20% for unseen final testing; remaining 80% is for 3-Fold CV
cv_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Class'], random_state=42)
cv_df = cv_df.reset_index(drop=True)
print(f"✅ Dataset ready! {len(cv_df)} samples allocated for 3-Fold Cross-Validation.")

# =====================================================================
# 2. DATASET CLASS & TRANSFORMS
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CTKidneyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')
        label = int(self.dataframe.iloc[idx]['target'])
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# =====================================================================
# 3. CONVNEXT TINY MODEL LOADER (WITH FROZEN BACKBONE)
# =====================================================================
def build_convnext_tiny(num_classes=4):
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    # Freeze pre-trained feature extractor backbone
    for param in model.features.parameters():
        param.requires_grad = False
    # Replace final linear classification layer
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    return model

# =====================================================================
# 4. 3-FOLD CROSS-VALIDATION EXECUTION (10 EPOCHS PER FOLD)
# =====================================================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
num_epochs = 10
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

convnext_results = []

print("\n" + "="*60)
print("🚀 INITIALIZING ARCHITECTURE: ConvNeXt Tiny (3-Fold CV)")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(cv_df, cv_df['Class']):
    print(f"\n>>> Starting Fold {fold}/3 for ConvNeXt Tiny <<<")
    print("-" * 50)

    fold_train_df = cv_df.iloc[train_idx]
    fold_val_df = cv_df.iloc[val_idx]

    fold_train_dataset = CTKidneyDataset(fold_train_df, transform=train_transforms)
    fold_val_dataset = CTKidneyDataset(fold_val_df, transform=test_val_transforms)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize fresh model for each fold
    model = build_convnext_tiny(num_classes=4).to(device)
    # Optimize only trainable parameters
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, weight_decay=1e-4)

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        train_loss, train_corrects, train_size = 0.0, 0, 0
        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_size += inputs.size(0)

        epoch_train_loss = train_loss / train_size
        epoch_train_acc = train_corrects.double().item() / train_size

        # Validation Phase
        model.eval()
        val_loss, val_corrects, val_size = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_size += inputs.size(0)

        epoch_val_loss = val_loss / val_size
        epoch_val_acc = val_corrects.double().item() / val_size

        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

        convnext_results.append({
            "Model": "ConvNeXt_Tiny",
            "Fold": fold,
            "Epoch": epoch + 1,
            "Train_Loss": epoch_train_loss,
            "Train_Acc": epoch_train_acc,
            "Val_Loss": epoch_val_loss,
            "Val_Acc": epoch_val_acc
        })

    fold += 1

# =====================================================================
# 5. CALCULATE STATISTICAL METRICS (MEAN, STD DEV, P-VALUE)
# =====================================================================
df_convnext = pd.DataFrame(convnext_results)

# Paired t-test across all 30 epoch data points (3 folds x 10 epochs)
t_stat, p_value = stats.ttest_rel(df_convnext['Train_Loss'], df_convnext['Val_Loss'])

print("\n" + "📊"*20)
print("CONVNEXT TINY 3-FOLD CV STATISTICAL METRICS")
print("📊"*20)
print(f"Model Name          : ConvNeXt Tiny")
print(f"Mean Val Accuracy   : {df_convnext['Val_Acc'].mean()*100:.2f}%")
print(f"Std Dev Val Acc     : {df_convnext['Val_Acc'].std():.4f}")
print(f"Mean Val Loss       : {df_convnext['Val_Loss'].mean():.4f}")
print(f"Std Dev Val Loss    : {df_convnext['Val_Loss'].std():.4f}")
print(f"p-value (Train/Val) : {p_value:.4e}")

# Save results to CSV
df_convnext.to_csv("convnext_tiny_3fold_results.csv", index=False)
print("\n✅ Results saved to 'convnext_tiny_3fold_results.csv'!")

📥 Downloading dataset via kagglehub...
✅ Dataset ready! 9956 samples allocated for 3-Fold Cross-Validation.

🚀 INITIALIZING ARCHITECTURE: ConvNeXt Tiny (3-Fold CV)

>>> Starting Fold 1/3 for ConvNeXt Tiny <<<
--------------------------------------------------
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:01<00:00, 112MB/s]


Epoch 01/10 | Train Loss: 0.7183 - Train Acc: 0.7445 | Val Loss: 0.4756 - Val Acc: 0.8132
Epoch 02/10 | Train Loss: 0.4265 - Train Acc: 0.8647 | Val Loss: 0.3359 - Val Acc: 0.8834
Epoch 03/10 | Train Loss: 0.3398 - Train Acc: 0.8900 | Val Loss: 0.2699 - Val Acc: 0.9096
Epoch 04/10 | Train Loss: 0.2938 - Train Acc: 0.9054 | Val Loss: 0.2378 - Val Acc: 0.9259
Epoch 05/10 | Train Loss: 0.2599 - Train Acc: 0.9156 | Val Loss: 0.2521 - Val Acc: 0.9078
Epoch 06/10 | Train Loss: 0.2336 - Train Acc: 0.9286 | Val Loss: 0.2080 - Val Acc: 0.9313
Epoch 07/10 | Train Loss: 0.2135 - Train Acc: 0.9316 | Val Loss: 0.2005 - Val Acc: 0.9325
Epoch 08/10 | Train Loss: 0.2062 - Train Acc: 0.9345 | Val Loss: 0.1796 - Val Acc: 0.9449
Epoch 09/10 | Train Loss: 0.1916 - Train Acc: 0.9355 | Val Loss: 0.1622 - Val Acc: 0.9497
Epoch 10/10 | Train Loss: 0.1786 - Train Acc: 0.9414 | Val Loss: 0.1572 - Val Acc: 0.9524

>>> Starting Fold 2/3 for ConvNeXt Tiny <<<
--------------------------------------------------
Epoc

In [3]:
!pip install -q kagglehub

import os
import glob
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy import stats
from PIL import Image
import kagglehub

# =====================================================================
# 1. DOWNLOAD DATASET & PREPARE SPLITS
# =====================================================================
print("📥 Downloading dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")

csv_search = glob.glob(f"{dataset_path}/**/*.csv", recursive=True)
df = pd.read_csv(csv_search[0])

all_image_paths = glob.glob(f"{dataset_path}/**/*.jpg", recursive=True)
path_dict = {os.path.basename(p): p for p in all_image_paths}
df['path'] = df['path'].apply(lambda x: path_dict.get(os.path.basename(str(x))))
df = df.dropna(subset=['path'])

# Keep 20% for unseen final testing; remaining 80% is for 3-Fold CV
cv_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Class'], random_state=42)
cv_df = cv_df.reset_index(drop=True)
print(f"✅ Dataset ready! {len(cv_df)} samples allocated for 3-Fold Cross-Validation.")

# =====================================================================
# 2. DATASET CLASS & TRANSFORMS
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CTKidneyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')
        label = int(self.dataframe.iloc[idx]['target'])
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# =====================================================================
# 3. VGG16 MODEL LOADER (WITH FROZEN FEATURE EXTRACTION BACKBONE)
# =====================================================================
def build_vgg16(num_classes=4):
    model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    # Freeze pre-trained convolutional feature extractor to prevent gradient explosion
    for param in model.features.parameters():
        param.requires_grad = False
    # Replace final classification linear layer
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    return model

# =====================================================================
# 4. 3-FOLD CROSS-VALIDATION EXECUTION (10 EPOCHS PER FOLD)
# =====================================================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
num_epochs = 10
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

vgg_results = []

print("\n" + "="*60)
print("🚀 INITIALIZING ARCHITECTURE: VGG16 (3-Fold CV)")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(cv_df, cv_df['Class']):
    print(f"\n>>> Starting Fold {fold}/3 for VGG16 <<<")
    print("-" * 50)

    fold_train_df = cv_df.iloc[train_idx]
    fold_val_df = cv_df.iloc[val_idx]

    fold_train_dataset = CTKidneyDataset(fold_train_df, transform=train_transforms)
    fold_val_dataset = CTKidneyDataset(fold_val_df, transform=test_val_transforms)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize fresh model for each fold
    model = build_vgg16(num_classes=4).to(device)
    # Optimize only trainable parameters
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, weight_decay=1e-4)

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        train_loss, train_corrects, train_size = 0.0, 0, 0
        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_size += inputs.size(0)

        epoch_train_loss = train_loss / train_size
        epoch_train_acc = train_corrects.double().item() / train_size

        # Validation Phase
        model.eval()
        val_loss, val_corrects, val_size = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_size += inputs.size(0)

        epoch_val_loss = val_loss / val_size
        epoch_val_acc = val_corrects.double().item() / val_size

        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

        vgg_results.append({
            "Model": "VGG16",
            "Fold": fold,
            "Epoch": epoch + 1,
            "Train_Loss": epoch_train_loss,
            "Train_Acc": epoch_train_acc,
            "Val_Loss": epoch_val_loss,
            "Val_Acc": epoch_val_acc
        })

    fold += 1

# =====================================================================
# 5. CALCULATE STATISTICAL METRICS (MEAN, STD DEV, P-VALUE)
# =====================================================================
df_vgg = pd.DataFrame(vgg_results)

# Paired t-test across all 30 epoch data points (3 folds x 10 epochs)
t_stat, p_value = stats.ttest_rel(df_vgg['Train_Loss'], df_vgg['Val_Loss'])

print("\n" + "📊"*20)
print("VGG16 3-FOLD CV STATISTICAL METRICS")
print("📊"*20)
print(f"Model Name          : VGG16")
print(f"Mean Val Accuracy   : {df_vgg['Val_Acc'].mean()*100:.2f}%")
print(f"Std Dev Val Acc     : {df_vgg['Val_Acc'].std():.4f}")
print(f"Mean Val Loss       : {df_vgg['Val_Loss'].mean():.4f}")
print(f"Std Dev Val Loss    : {df_vgg['Val_Loss'].std():.4f}")
print(f"p-value (Train/Val) : {p_value:.4e}")

# Save results to CSV
df_vgg.to_csv("vgg16_3fold_results.csv", index=False)
print("\n✅ Results saved to 'vgg16_3fold_results.csv'!")

📥 Downloading dataset via kagglehub...
Using Colab cache for faster access to the 'ct-kidney-dataset-normal-cyst-tumor-and-stone' dataset.
✅ Dataset ready! 9956 samples allocated for 3-Fold Cross-Validation.

🚀 INITIALIZING ARCHITECTURE: VGG16 (3-Fold CV)

>>> Starting Fold 1/3 for VGG16 <<<
--------------------------------------------------
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:08<00:00, 68.4MB/s]


Epoch 01/10 | Train Loss: 0.8835 - Train Acc: 0.7219 | Val Loss: 0.3621 - Val Acc: 0.9187
Epoch 02/10 | Train Loss: 0.6161 - Train Acc: 0.8566 | Val Loss: 0.1889 - Val Acc: 0.9364
Epoch 03/10 | Train Loss: 0.4892 - Train Acc: 0.8915 | Val Loss: 0.1805 - Val Acc: 0.9473
Epoch 04/10 | Train Loss: 0.3602 - Train Acc: 0.9141 | Val Loss: 0.2548 - Val Acc: 0.9479
Epoch 05/10 | Train Loss: 0.4093 - Train Acc: 0.9161 | Val Loss: 0.2598 - Val Acc: 0.9277
Epoch 06/10 | Train Loss: 0.3173 - Train Acc: 0.9349 | Val Loss: 0.1278 - Val Acc: 0.9647
Epoch 07/10 | Train Loss: 0.4304 - Train Acc: 0.9257 | Val Loss: 0.0929 - Val Acc: 0.9723
Epoch 08/10 | Train Loss: 0.3066 - Train Acc: 0.9426 | Val Loss: 0.1302 - Val Acc: 0.9717
Epoch 09/10 | Train Loss: 0.4774 - Train Acc: 0.9336 | Val Loss: 0.1123 - Val Acc: 0.9729
Epoch 10/10 | Train Loss: 0.3798 - Train Acc: 0.9420 | Val Loss: 0.1409 - Val Acc: 0.9801

>>> Starting Fold 2/3 for VGG16 <<<
--------------------------------------------------
Epoch 01/10 